In [1]:
# Cell 1 — Imports, paths & helpers
import os, re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Import Random Forest
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# --- Paths ---
DATA_MERGED   = Path("./data_thuydien/merged_hoa_binh_keyjoin.csv")  # dùng để TRAIN
DATA_ENRICHED = Path("./data_thuydien/data_thuydien_enriched.csv")   # dùng để DEMO/INFERENCE

# Đổi folder output sang 'rf'
OUT_DIR     = Path("./checkpoint/rf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH  = OUT_DIR / "rf_model.pkl"
FEATS_PATH  = OUT_DIR / "features_used.json"
META_PATH   = OUT_DIR / "train_meta.json"
PRED_PATH   = OUT_DIR / "predictions_test.csv"
FIG_TS_PATH = OUT_DIR / "plot_test_timeseries.png"
FIG_PP_PATH = OUT_DIR / "plot_test_parity.png"
FIG_FI_PATH = OUT_DIR / "plot_feature_importance.png"

# --- Helpers (Giữ nguyên) ---
def normalize_col(c: str) -> str:
    c0 = re.sub(r"\s+", " ", str(c).strip().lower())
    c0 = re.sub(r"\s*\([^)]*\)", "", c0)
    c0 = c0.replace("%", "pct").replace("°", "")
    c0 = re.sub(r"[^a-z0-9_ ]+", "_", c0).replace(" ", "_")
    c0 = re.sub(r"_+", "_", c0).strip("_")
    return c0

def find_first_col(cols, *keywords):
    for c in cols:
        low = c.lower()
        if all(k in low for k in keywords):
            return c
    return None

def parse_best_datetime(series):
    s1 = pd.to_datetime(series, errors="coerce", dayfirst=False)
    s2 = pd.to_datetime(series, errors="coerce", dayfirst=True)
    return s1 if s1.notna().sum() >= s2.notna().sum() else s2

def to_numeric_safe(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace(",", ".", regex=False).str.replace(" ", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

print("✓ Helpers ready. Model: Random Forest")

✓ Helpers ready. Model: Random Forest


In [2]:
# Cell 2 — Load (TRAIN) merged & detect columns
assert DATA_MERGED.exists(), f"Không tìm thấy file TRAIN: {DATA_MERGED}"
raw = pd.read_csv(DATA_MERGED)

norm_map = {c: normalize_col(c) for c in raw.columns}
df = raw.rename(columns=norm_map).copy()

# Tìm cột thời gian & các cột chính
time_col = "thoi_diem" if "thoi_diem" in df.columns else ("time" if "time" in df.columns else None)
assert time_col is not None, f"Thiếu cột thời gian trong merged: {df.columns.tolist()}"

col_target = find_first_col(df.columns, "muc","nuoc","thuong","luu")
col_flow   = "luu_luong_den_ho_m3_s_capped" if "luu_luong_den_ho_m3_s_capped" in df.columns else (
             "luu_luong_den_ho_m3_s" if "luu_luong_den_ho_m3_s" in df.columns else
             find_first_col(df.columns, "luu","luong","den","ho"))

# Thời tiết
col_temp = "temp_c"    if "temp_c"    in df.columns else find_first_col(df.columns, "temp","temperature")
col_rh   = "rh_pct"    if "rh_pct"    in df.columns else find_first_col(df.columns, "rh","humidity")
col_pr   = "precip_mm" if "precip_mm" in df.columns else find_first_col(df.columns, "precip","mua")
col_cc   = "cloud_pct" if "cloud_pct" in df.columns else find_first_col(df.columns, "cloud")

print("TRAIN columns (from merged):")
print(" - time_col:", time_col)
print(" - target  :", col_target)
print(" - inflow  :", col_flow)
print(" - weather :", [col_temp, col_rh, col_pr, col_cc])

# Parse time & sort
dt = parse_best_datetime(df[time_col])
df = df.assign(thoi_diem=dt).dropna(subset=["thoi_diem"]).sort_values("thoi_diem").reset_index(drop=True)
if time_col != "thoi_diem":
    df = df.drop(columns=[time_col])

keep = ["thoi_diem"] + [c for c in [col_target, col_flow, col_temp, col_rh, col_pr, col_cc] if c is not None]
df = df[keep].copy()

for c in keep:
    if c == "thoi_diem": continue
    df[c] = to_numeric_safe(df[c])

print("[TRAIN] Shape after select:", df.shape)
df.head(3)

TRAIN columns (from merged):
 - time_col: thoi_diem
 - target  : muc_nuoc_thuong_luu_m
 - inflow  : luu_luong_den_ho_m3_s_capped
 - weather : ['temp_c', 'rh_pct', 'precip_mm', 'cloud_pct']
[TRAIN] Shape after select: (33264, 7)


,thoi_diem,muc_nuoc_thuong_luu_m,luu_luong_den_ho_m3_s_capped,temp_c,rh_pct,precip_mm,cloud_pct
0,2022-01-01 00:00:00,112.13,848.0,14.5,94,0.0,65
1,2022-01-01 01:00:00,112.14,700.0,14.6,94,0.0,100
2,2022-01-01 02:00:00,112.14,700.0,14.9,92,0.0,100


In [3]:
# Cell 3 — Feature engineering (TRAIN, strictly-past 28 ngày)
WEATHER_COLS = [c for c in [col_temp, col_rh, col_pr, col_cc] if c is not None]
VAR_LIST = [c for c in [col_target, col_flow] + WEATHER_COLS if c is not None]

df = df.set_index("thoi_diem").sort_index()

# Time features
feat = pd.DataFrame(index=df.index)
feat["hour"] = feat.index.hour
feat["dow"]  = feat.index.dayofweek
feat["month"]= feat.index.month
feat["is_weekend"] = (feat["dow"] >= 5).astype(int)

def add_roll_feats(dst: pd.DataFrame, src: pd.Series, name: str, windows=("24H","3D","7D","14D","28D")):
    s = src.shift(1)  # tránh leakage
    for w in windows:
        r = s.rolling(w, closed="left")
        dst[f"{name}__mean_{w.lower()}"] = r.mean()
        dst[f"{name}__std_{w.lower()}"]  = r.std()
        dst[f"{name}__max_{w.lower()}"]  = r.max()
        dst[f"{name}__min_{w.lower()}"]  = r.min()

def add_lag_feats(dst: pd.DataFrame, src: pd.Series, name: str, lags=(1,3,6,12,24,48,72)):
    for k in lags:
        dst[f"{name}__lag_{k}"] = src.shift(k)

for c in VAR_LIST:
    add_roll_feats(feat, df[c], c)
    add_lag_feats(feat, df[c], c)

# Target
y = df[col_target].copy()

# Ghép features + target
data = pd.concat([feat, y.rename("target")], axis=1).dropna(subset=["target"])

print("Feature matrix:", feat.shape, "| Data rows:", data.shape[0])
print("Example columns:", list(feat.columns)[:10])

Feature matrix: (33264, 166) | Data rows: 33264
Example columns: ['hour', 'dow', 'month', 'is_weekend', 'muc_nuoc_thuong_luu_m__mean_24h', 'muc_nuoc_thuong_luu_m__std_24h', 'muc_nuoc_thuong_luu_m__max_24h', 'muc_nuoc_thuong_luu_m__min_24h', 'muc_nuoc_thuong_luu_m__mean_3d', 'muc_nuoc_thuong_luu_m__std_3d']


In [4]:
# Cell 4 — Split 80/20 theo thời gian
N = len(data)
split_idx = int(N * 0.8)
train = data.iloc[:split_idx].copy()
test  = data.iloc[split_idx:].copy()

X_train, y_train = train.drop(columns=["target"]), train["target"]
X_test,  y_test  = test.drop(columns=["target"]),  test["target"]

print(f"Tổng số mẫu: {N:,} -> Train: {len(train):,} | Test: {len(test):,}")
print(f"Số feature: {X_train.shape[1]}")
print("Khoảng thời gian train:", X_train.index.min(), "->", X_train.index.max())
print("Khoảng thời gian test :", X_test.index.min(),  "->", X_test.index.max())

Tổng số mẫu: 33,264 -> Train: 26,611 | Test: 6,653
Số feature: 166
Khoảng thời gian train: 2022-01-01 00:00:00 -> 2025-01-13 18:00:00
Khoảng thời gian test : 2025-01-13 19:00:00 -> 2025-10-17 23:00:00


In [5]:
# Cell 5 — Train Random Forest
# Random Forest của sklearn cần xử lý NaN
X_train_clean = X_train.fillna(0)

# Cấu hình Random Forest
# n_estimators: số lượng cây (ví dụ 200)
# max_depth: độ sâu tối đa (để None hoặc số cụ thể như 20 để tránh overfitting quá mức)
# n_jobs=-1: sử dụng tất cả CPU core
model = RandomForestRegressor(
    n_estimators=200, 
    max_depth=20,       
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Start training Random Forest...")
model.fit(X_train_clean, y_train)
print("Random Forest Model trained.")

Start training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   10.6s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  1.1min


✓ Random Forest Model trained.


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  1.2min finished


In [6]:
# Cell 6 — Evaluate & save artifacts
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return (np.abs((y_true[mask]-y_pred[mask]) / y_true[mask])).mean() * 100

# Clean data test
X_test_clean = X_test.fillna(0)

yhat_tr = model.predict(X_train_clean)
yhat_te = model.predict(X_test_clean)

metrics = {
    "train": {"MAE": float(mean_absolute_error(y_train, yhat_tr)),
              "RMSE": float(mean_squared_error(y_train, yhat_tr)),
              "R2": float(r2_score(y_train, yhat_tr)),
              "MAPE_pct": float(mape(y_train, yhat_tr))},
    "test":  {"MAE": float(mean_absolute_error(y_test, yhat_te)),
              "RMSE": float(mean_squared_error(y_test, yhat_te)),
              "R2": float(r2_score(y_test, yhat_te)),
              "MAPE_pct": float(mape(y_test, yhat_te))},
    "model_type": "RandomForestRegressor",
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Save model + features (dùng joblib cho sklearn models)
joblib.dump(model, MODEL_PATH)

with open(FEATS_PATH, "w", encoding="utf-8") as f:
    json.dump(X_train.columns.tolist(), f, ensure_ascii=False, indent=2)

pred_df = pd.DataFrame({"thoi_diem": X_test.index, "y_true": y_test.values, "y_pred": yhat_te})
pred_df.to_csv(PRED_PATH, index=False)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Saved:")
print(" - MODEL_PATH:", MODEL_PATH)
print(" - FEATS_PATH:", FEATS_PATH)
print(" - META_PATH :", META_PATH)
print(" - PRED_PATH :", PRED_PATH)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished


{
  "train": {
    "MAE": 0.005999291142562702,
    "RMSE": 0.001681057443312434,
    "R2": 0.9999412643112636,
    "MAPE_pct": 0.005468613810588609
  },
  "test": {
    "MAE": 1.3922834563361626,
    "RMSE": 10.725710629673376,
    "R2": 0.8618739760189894,
    "MAPE_pct": 1.5839100024711599
  },
  "model_type": "RandomForestRegressor"
}
Saved:
 - MODEL_PATH: checkpoint\rf\rf_model.pkl
 - FEATS_PATH: checkpoint\rf\features_used.json
 - META_PATH : checkpoint\rf\train_meta.json
 - PRED_PATH : checkpoint\rf\predictions_test.csv


In [7]:
# Cell 7 — Feature Importance
if hasattr(model, "feature_importances_"):
    fi_df = pd.DataFrame({
        "feature": X_train.columns, 
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    
    top_k = min(30, len(fi_df))
    plt.figure(figsize=(10, max(4, int(0.25*top_k)+2)))
    plt.barh(fi_df["feature"].head(top_k)[::-1], fi_df["importance"].head(top_k)[::-1])
    plt.xlabel("Importance"); plt.title("Top Feature Importance (Random Forest)")
    plt.tight_layout(); plt.savefig(FIG_FI_PATH, dpi=150); plt.close()
    print("Saved feature importance:", FIG_FI_PATH)
else:
    print("No feature importance available.")

Saved feature importance: checkpoint\rf\plot_feature_importance.png


In [8]:
# Cell 8 — Plots (time-series + parity)
# Time-series 500 điểm cuối
k = 500
sl = slice(max(0, len(pred_df)-k), len(pred_df))
plt.figure(figsize=(12,4))
plt.plot(pred_df["thoi_diem"].values[sl], pred_df["y_true"].values[sl], label="Thực tế")
plt.plot(pred_df["thoi_diem"].values[sl], pred_df["y_pred"].values[sl], label="Dự báo")
plt.title("Test — y_true vs y_pred (Random Forest - 500 điểm cuối)")
plt.xlabel("Thời điểm"); plt.ylabel("Mực nước (m)")
plt.xticks(rotation=25); plt.legend(); plt.tight_layout()
plt.savefig(FIG_TS_PATH, dpi=150); plt.close()

# Parity
plt.figure(figsize=(5,5))
plt.scatter(pred_df["y_true"].values, pred_df["y_pred"].values, s=6, alpha=0.6)
mn = float(min(pred_df["y_true"].min(), pred_df["y_pred"].min()))
mx = float(max(pred_df["y_true"].max(), pred_df["y_pred"].max()))
plt.plot([mn, mx], [mn, mx], color='red')
plt.title("Parity plot — Test")
plt.xlabel("y_true (m)"); plt.ylabel("y_pred (m)")
plt.tight_layout()
plt.savefig(FIG_PP_PATH, dpi=150); plt.close()

print("Saved figures:")
print(" - TS:", FIG_TS_PATH)
print(" - PP:", FIG_PP_PATH)

Saved figures:
 - TS: checkpoint\rf\plot_test_timeseries.png
 - PP: checkpoint\rf\plot_test_parity.png


In [9]:
# Cell 9 — Đọc ENRICHED & chuẩn hoá cột (cho inference)
assert DATA_ENRICHED.exists(), f"Không tìm thấy file DEMO: {DATA_ENRICHED}"
enr_raw = pd.read_csv(DATA_ENRICHED)

# Chuẩn tên cột
norm_map_en = {c: normalize_col(c) for c in enr_raw.columns}
df_en = enr_raw.rename(columns=norm_map_en).copy()

# Cột thời gian
time_candidates = [c for c in ["thoi_diem","time","datetime","timestamp"] if c in df_en.columns]
assert len(time_candidates) > 0, f"Thiếu cột thời gian trong enriched: {df_en.columns.tolist()}"
time_col_en = time_candidates[0]

# Parse & sort
dt_en = parse_best_datetime(df_en[time_col_en])
df_en = df_en.assign(thoi_diem=dt_en).dropna(subset=["thoi_diem"]).sort_values("thoi_diem").reset_index(drop=True)
if time_col_en != "thoi_diem":
    df_en = df_en.drop(columns=[time_col_en])

# Map về tên chuẩn
rename_map = {}
if "temperature_2m" in df_en.columns:       rename_map["temperature_2m"]       = "temp_c"
if "relative_humidity_2m" in df_en.columns: rename_map["relative_humidity_2m"] = "rh_pct"
if "precipitation" in df_en.columns:        rename_map["precipitation"]        = "precip_mm"
if "cloud_cover" in df_en.columns:          rename_map["cloud_cover"]          = "cloud_pct"
df_en = df_en.rename(columns=rename_map)

# Cột chính
col_target_en = "muc_nuoc_thuong_luu_m" if "muc_nuoc_thuong_luu_m" in df_en.columns else None
col_flow_en   = "luu_luong_den_ho_m3_s_capped" if "luu_luong_den_ho_m3_s_capped" in df_en.columns else (
                "luu_luong_den_ho_m3_s" if "luu_luong_den_ho_m3_s" in df_en.columns else None)
col_temp_en = "temp_c"    if "temp_c"    in df_en.columns else None
col_rh_en   = "rh_pct"    if "rh_pct"    in df_en.columns else None
col_pr_en   = "precip_mm" if "precip_mm" in df_en.columns else None
col_cc_en   = "cloud_pct" if "cloud_pct" in df_en.columns else None

VAR_LIST_INF = [v for v in [col_target_en or col_target, col_flow_en or col_flow,
                            col_temp_en, col_rh_en, col_pr_en, col_cc_en] if v is not None]

need_cols_en = ["thoi_diem"] + VAR_LIST_INF
df_en = df_en[need_cols_en].copy()

for c in need_cols_en:
    if c == "thoi_diem": continue
    df_en[c] = to_numeric_safe(df_en[c])

df_en = df_en.set_index("thoi_diem").sort_index()
print("[DEMO] enriched range:", df_en.index.min(), "->", df_en.index.max())
print("[DEMO] columns used:", VAR_LIST_INF)

[DEMO] enriched range: 2025-09-01 00:00:00 -> 2025-10-23 23:00:00
[DEMO] columns used: ['muc_nuoc_thuong_luu_m', 'luu_luong_den_ho_m3_s', 'temp_c', 'rh_pct', 'precip_mm', 'cloud_pct']


In [10]:
# Cell 10 — Inference: “anchor vào bản ghi gần nhất trước t” với enriched
TARGET_TIME = "2025-10-25 07:00" 

# Load model (joblib) & feature list
inf_model = joblib.load(MODEL_PATH)

with open(FEATS_PATH, "r", encoding="utf-8") as f:
    feat_list = json.load(f)

try:
    VAR_LIST  # from TRAIN
    var_list_infer = VAR_LIST
except NameError:
    var_list_infer = VAR_LIST_INF

def build_features_at_time_from_enriched(frame: pd.DataFrame, t: pd.Timestamp,
                                         vars_used: list,
                                         min_days_coverage: int = 0) -> pd.DataFrame:
    t = pd.to_datetime(t)
    idx = frame.index[frame.index < t]
    if len(idx) == 0:
        raise ValueError("Enriched không có quan sát nào trước thời điểm yêu cầu.")
    t_anchor = idx.max()

    sub = frame.loc[:t_anchor].iloc[:-1]

    if min_days_coverage > 0:
        left_edge = t_anchor - pd.Timedelta(days=28)
        coverage_cnt = sub.loc[left_edge:t_anchor].shape[0]
        if coverage_cnt < min_days_coverage:
            print(f"[WARN] Vùng 28D trước anchor có {coverage_cnt} bản ghi (<{min_days_coverage}).")

    out = {}
    out["hour"] = t.hour
    out["dow"] = t.dayofweek
    out["month"] = t.month
    out["is_weekend"] = int(out["dow"] >= 5)

    def roll_stats(s: pd.Series, prefix: str):
        r24  = s.rolling("24H", closed="left").agg(["mean","std","max","min"])
        r3d  = s.rolling("3D",  closed="left").agg(["mean","std","max","min"])
        r7d  = s.rolling("7D",  closed="left").agg(["mean","std","max","min"])
        r14d = s.rolling("14D", closed="left").agg(["mean","std","max","min"])
        r28d = s.rolling("28D", closed="left").agg(["mean","std","max","min"])
        last = {}
        for name, robj in [("24h", r24), ("3d", r3d), ("7d", r7d), ("14d", r14d), ("28d", r28d)]:
            last[f"{prefix}__mean_{name}"] = robj["mean"].iloc[-1] if len(robj) else np.nan
            last[f"{prefix}__std_{name}"]  = robj["std"].iloc[-1]  if len(robj) else np.nan
            last[f"{prefix}__max_{name}"]  = robj["max"].iloc[-1]  if len(robj) else np.nan
            last[f"{prefix}__min_{name}"]  = robj["min"].iloc[-1]  if len(robj) else np.nan
        return last

    def add_lags(s: pd.Series, prefix: str, lags=(1,3,6,12,24,48,72)):
        for k in lags:
            out[f"{prefix}__lag_{k}"] = s.shift(k).iloc[-1] if len(s) >= k+1 else np.nan

    for col in vars_used:
        if col not in frame.columns:
            for w in ["24h","3d","7d","14d","28d"]:
                out[f"{col}__mean_{w}"] = np.nan
                out[f"{col}__std_{w}"]  = np.nan
                out[f"{col}__max_{w}"]  = np.nan
                out[f"{col}__min_{w}"]  = np.nan
            for k in [1,3,6,12,24,48,72]:
                out[f"{col}__lag_{k}"] = np.nan
            continue

        s = frame[col].astype(float)
        s.index = pd.to_datetime(frame.index)
        s = s.loc[:t_anchor].iloc[:-1]
        out.update(roll_stats(s, col))
        add_lags(s, col)

    row = pd.DataFrame({k:[v] for k,v in out.items()})
    for c in feat_list:
        if c not in row.columns:
            row[c] = np.nan
    row = row[feat_list]
    return row

# Parse TARGET_TIME
t = pd.to_datetime(TARGET_TIME, dayfirst=True, errors="coerce")
if pd.isna(t):
    t = pd.to_datetime(TARGET_TIME)

rowX = build_features_at_time_from_enriched(df_en, t, var_list_infer, min_days_coverage=0)

# Random Forest cần fillna
rowX_clean = rowX.fillna(0)

yhat = float(inf_model.predict(rowX_clean)[0])
print(f"[Predict @ {t}] (Model: Random Forest) => Mực nước TL ước lượng: {yhat:.4f} m")

[Predict @ 2025-10-25 07:00:00] (Model: Random Forest) => Mực nước TL ước lượng: 115.6757 m


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished
